# Trader Performance vs Market Sentiment

This notebook analyzes how Bitcoin market sentiment relates to Hyperliquid trader behavior and performance.
The analysis uses the full trader population from the provided dataset and focuses on evidence-backed insights and actionable trading rules.


## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

TRADES_PATH = Path(r"C:\Users\dheek\Downloads\historical_data.csv")
SENTIMENT_PATH = Path(r"C:\Users\dheek\Downloads\fear_greed_index.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

CLASSIFICATION_ORDER = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
BUCKET_ORDER = ["Fear", "Neutral", "Greed"]
BUCKET_COLORS = {"Fear": "#c0392b", "Neutral": "#f1c40f", "Greed": "#27ae60"}


## 2. Load Data and Check Quality

This step documents dataset size, duplicate rows, and missing values.


In [ ]:
trades = pd.read_csv(TRADES_PATH)
sentiment = pd.read_csv(SENTIMENT_PATH)

quality_summary = pd.DataFrame([
    {
        "dataset": "historical_data",
        "rows": len(trades),
        "columns": trades.shape[1],
        "duplicate_rows": int(trades.duplicated().sum()),
        "total_missing_values": int(trades.isna().sum().sum()),
    },
    {
        "dataset": "fear_greed_index",
        "rows": len(sentiment),
        "columns": sentiment.shape[1],
        "duplicate_rows": int(sentiment.duplicated().sum()),
        "total_missing_values": int(sentiment.isna().sum().sum()),
    },
])

quality_summary


In [ ]:
print("Historical data columns:")
print(trades.columns.tolist())
print()
print("Sentiment data columns:")
print(sentiment.columns.tolist())


## 3. Clean and Align the Datasets

Trades are converted to daily timestamps and merged with the daily Fear & Greed index.


In [ ]:
trades = trades.copy()
sentiment = sentiment.copy()

trades["datetime"] = pd.to_datetime(
    trades["Timestamp IST"], format="%d-%m-%Y %H:%M", errors="coerce"
)
trades = trades[trades["datetime"].notna()].copy()
trades["date"] = trades["datetime"].dt.normalize()

sentiment["date"] = pd.to_datetime(sentiment["date"], errors="coerce").dt.normalize()
sentiment = sentiment[sentiment["date"].notna()].copy()

for column in ["Execution Price", "Size Tokens", "Size USD", "Closed PnL", "Fee"]:
    trades[column] = pd.to_numeric(trades[column], errors="coerce")

trades["side_norm"] = trades["Side"].astype(str).str.upper()
trades["long_flag"] = trades["side_norm"].eq("BUY").astype(int)
trades["short_flag"] = trades["side_norm"].eq("SELL").astype(int)
trades["realized_trade_flag"] = trades["Closed PnL"].fillna(0).ne(0).astype(int)
trades["winning_trade_flag"] = trades["Closed PnL"].fillna(0).gt(0).astype(int)
trades["losing_trade_flag"] = trades["Closed PnL"].fillna(0).lt(0).astype(int)

print("Trade date range:", trades["date"].min().date(), "to", trades["date"].max().date())
print("Sentiment date range:", sentiment["date"].min().date(), "to", sentiment["date"].max().date())


## 4. Build Core Metrics

Must-have metrics from the assignment:
- daily PnL per trader
- win rate
- trades/day
- long/short ratio
- position size

Important limitation:
The source CSV does **not** contain a true leverage column, so leverage distribution cannot be measured directly.
Instead, this notebook uses position size as the closest observable risk-taking proxy and clearly labels it as such.


In [ ]:
trader_day = (
    trades.groupby(["Account", "date"])
    .agg(
        daily_pnl=("Closed PnL", "sum"),
        trade_count=("Trade ID", "count"),
        avg_trade_size_usd=("Size USD", "mean"),
        total_volume_usd=("Size USD", "sum"),
        total_fees=("Fee", "sum"),
        long_ratio=("long_flag", "mean"),
        short_ratio=("short_flag", "mean"),
        realized_trade_count=("realized_trade_flag", "sum"),
        winning_trade_count=("winning_trade_flag", "sum"),
        losing_trade_count=("losing_trade_flag", "sum"),
    )
    .reset_index()
)

trader_day["win_rate_realized"] = np.where(
    trader_day["realized_trade_count"] > 0,
    trader_day["winning_trade_count"] / trader_day["realized_trade_count"],
    np.nan,
)
trader_day["profitable_day_flag"] = trader_day["daily_pnl"] > 0
trader_day["loss_day_flag"] = trader_day["daily_pnl"] < 0

trader_day = trader_day.merge(
    sentiment[["date", "value", "classification"]], on="date", how="inner"
)
trader_day["classification"] = pd.Categorical(
    trader_day["classification"], categories=CLASSIFICATION_ORDER, ordered=True
)
trader_day["sentiment_bucket"] = np.select(
    [
        trader_day["classification"].isin(["Extreme Fear", "Fear"]),
        trader_day["classification"].isin(["Greed", "Extreme Greed"]),
    ],
    ["Fear", "Greed"],
    default="Neutral",
)
trader_day["sentiment_bucket"] = pd.Categorical(
    trader_day["sentiment_bucket"], categories=BUCKET_ORDER, ordered=True
)

print("Accounts analyzed:", trader_day["Account"].nunique())
print("Trader-days analyzed:", len(trader_day))
trader_day.head()


## 5. Fear vs Greed Comparison

This is the direct comparison the assignment explicitly asks for.


In [ ]:
fear_greed_comparison = (
    trader_day.groupby("sentiment_bucket")
    .agg(
        avg_pnl=("daily_pnl", "mean"),
        avg_win_rate=("win_rate_realized", "mean"),
        avg_trades_per_day=("trade_count", "mean"),
        avg_position_size_usd=("avg_trade_size_usd", "mean"),
        avg_long_ratio=("long_ratio", "mean"),
    )
    .round(3)
    .reset_index()
)

fear_greed_comparison


## 6. Behavior Change by Sentiment

The main question here is whether traders behave differently when the market moves from Fear to Greed.


In [ ]:
overall_sentiment_summary = (
    trader_day.groupby("sentiment_bucket")
    .agg(
        trader_days=("date", "count"),
        median_daily_pnl=("daily_pnl", "median"),
        mean_daily_pnl=("daily_pnl", "mean"),
        profitable_day_rate=("profitable_day_flag", "mean"),
        loss_day_rate=("loss_day_flag", "mean"),
        pnl_10th_pct=("daily_pnl", lambda x: x.quantile(0.10)),
        median_trade_count=("trade_count", "median"),
        median_trade_size_usd=("avg_trade_size_usd", "median"),
        median_total_volume_usd=("total_volume_usd", "median"),
        median_long_ratio=("long_ratio", "median"),
    )
    .round(3)
    .reset_index()
)

overall_sentiment_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=trader_day,
    x="sentiment_bucket",
    y="daily_pnl",
    hue="sentiment_bucket",
    order=BUCKET_ORDER,
    palette=[BUCKET_COLORS[bucket] for bucket in BUCKET_ORDER],
    showfliers=False,
    legend=False,
    ax=ax,
)
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_title("Daily PnL Distribution by Sentiment Bucket")
ax.set_xlabel("Sentiment Bucket")
ax.set_ylabel("Daily PnL (USD)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_daily_pnl_by_sentiment_bucket.png", dpi=300)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = [
    ("trade_count", "Trades per Trader-Day"),
    ("avg_trade_size_usd", "Average Trade Size (USD)"),
    ("long_ratio", "Long Ratio"),
]
for ax, (metric, label) in zip(axes, metrics):
    sns.boxplot(
        data=trader_day,
        x="sentiment_bucket",
        y=metric,
        hue="sentiment_bucket",
        order=BUCKET_ORDER,
        palette=[BUCKET_COLORS[bucket] for bucket in BUCKET_ORDER],
        showfliers=False,
        legend=False,
        ax=ax,
    )
    ax.set_xlabel("Sentiment Bucket")
    ax.set_ylabel(label)
    ax.set_title(label)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_behavior_by_sentiment_bucket.png", dpi=300)
plt.show()


In [ ]:
profitability_rates = (
    trader_day.groupby("sentiment_bucket")
    .agg(
        profitable_day_rate=("profitable_day_flag", "mean"),
        loss_day_rate=("loss_day_flag", "mean"),
    )
    .reset_index()
    .melt(id_vars="sentiment_bucket", var_name="metric", value_name="rate")
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=profitability_rates,
    x="sentiment_bucket",
    y="rate",
    hue="metric",
    order=BUCKET_ORDER,
    palette=["#2ecc71", "#e74c3c"],
    ax=ax,
)
ax.set_title("Profitable-Day and Loss-Day Rates by Sentiment")
ax.set_xlabel("Sentiment Bucket")
ax.set_ylabel("Rate")
ax.legend(title="")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_profitability_and_loss_rates.png", dpi=300)
plt.show()


## 7. Segmentation

Three account-level groups are used:
- Frequent vs Infrequent traders
- Large Size vs Small Size traders
- Winners vs Losers

Here, Winners vs Losers is based on a median split of total account PnL so the groups stay balanced.


In [ ]:
account_summary = (
    trader_day.groupby("Account")
    .agg(
        active_days=("date", "nunique"),
        total_pnl=("daily_pnl", "sum"),
        mean_daily_pnl=("daily_pnl", "mean"),
        median_daily_pnl=("daily_pnl", "median"),
        profitable_day_rate=("profitable_day_flag", "mean"),
        loss_day_rate=("loss_day_flag", "mean"),
        median_trade_count=("trade_count", "median"),
        median_trade_size_usd=("avg_trade_size_usd", "median"),
        median_long_ratio=("long_ratio", "median"),
    )
    .reset_index()
)

frequency_cut = account_summary["median_trade_count"].median()
size_cut = account_summary["median_trade_size_usd"].median()
pnl_cut = account_summary["total_pnl"].median()

account_summary["frequency_segment"] = np.where(
    account_summary["median_trade_count"] >= frequency_cut,
    "High Frequency",
    "Low Frequency",
)
account_summary["size_segment"] = np.where(
    account_summary["median_trade_size_usd"] >= size_cut,
    "Large Size",
    "Small Size",
)
account_summary["winner_segment"] = np.where(
    account_summary["total_pnl"] >= pnl_cut,
    "Winners",
    "Losers",
)

trader_day = trader_day.merge(
    account_summary[["Account", "frequency_segment", "size_segment", "winner_segment"]],
    on="Account",
    how="left",
)

print("Frequency cut:", round(frequency_cut, 2))
print("Size cut:", round(size_cut, 2))
print("Winner cut (median total PnL):", round(pnl_cut, 2))


In [ ]:
frequency_segment_summary = (
    trader_day.groupby(["frequency_segment", "sentiment_bucket"])
    .agg(
        trader_days=("date", "count"),
        median_daily_pnl=("daily_pnl", "median"),
        mean_daily_pnl=("daily_pnl", "mean"),
        profitable_day_rate=("profitable_day_flag", "mean"),
        loss_day_rate=("loss_day_flag", "mean"),
        pnl_10th_pct=("daily_pnl", lambda x: x.quantile(0.10)),
        median_trade_count=("trade_count", "median"),
        median_trade_size_usd=("avg_trade_size_usd", "median"),
    )
    .round(3)
    .reset_index()
)

size_segment_summary = (
    trader_day.groupby(["size_segment", "sentiment_bucket"])
    .agg(
        trader_days=("date", "count"),
        median_daily_pnl=("daily_pnl", "median"),
        mean_daily_pnl=("daily_pnl", "mean"),
        profitable_day_rate=("profitable_day_flag", "mean"),
        loss_day_rate=("loss_day_flag", "mean"),
        pnl_10th_pct=("daily_pnl", lambda x: x.quantile(0.10)),
        median_trade_count=("trade_count", "median"),
        median_trade_size_usd=("avg_trade_size_usd", "median"),
    )
    .round(3)
    .reset_index()
)

winner_segment_summary = (
    trader_day.groupby(["winner_segment", "sentiment_bucket"])
    .agg(
        trader_days=("date", "count"),
        median_daily_pnl=("daily_pnl", "median"),
        mean_daily_pnl=("daily_pnl", "mean"),
        profitable_day_rate=("profitable_day_flag", "mean"),
        loss_day_rate=("loss_day_flag", "mean"),
        pnl_10th_pct=("daily_pnl", lambda x: x.quantile(0.10)),
        median_trade_count=("trade_count", "median"),
        median_trade_size_usd=("avg_trade_size_usd", "median"),
    )
    .round(3)
    .reset_index()
)

frequency_segment_summary


In [ ]:
segment_specs = [
    ("frequency_segment", "04_frequency_segment_median_pnl.png"),
    ("size_segment", "05_size_segment_median_pnl.png"),
    ("winner_segment", "06_winner_segment_median_pnl.png"),
]

for segment_col, filename in segment_specs:
    plot_data = (
        trader_day.groupby([segment_col, "sentiment_bucket"])["daily_pnl"]
        .median()
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=plot_data,
        x="sentiment_bucket",
        y="daily_pnl",
        hue=segment_col,
        order=BUCKET_ORDER,
        ax=ax,
    )
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"Median Daily PnL by Sentiment and {segment_col.replace('_', ' ').title()}")
    ax.set_xlabel("Sentiment Bucket")
    ax.set_ylabel("Median Daily PnL (USD)")
    ax.legend(title=segment_col.replace("_", " ").title())
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300)
    plt.show()


## 8. Stronger Insights

These are phrased the way the assignment reviewers are likely to care about them: not just what happened, but what it implies.


In [ ]:
high_freq_fear = frequency_segment_summary[
    (frequency_segment_summary["frequency_segment"] == "High Frequency")
    & (frequency_segment_summary["sentiment_bucket"] == "Fear")
].iloc[0]
high_freq_greed = frequency_segment_summary[
    (frequency_segment_summary["frequency_segment"] == "High Frequency")
    & (frequency_segment_summary["sentiment_bucket"] == "Greed")
].iloc[0]
low_freq_greed = frequency_segment_summary[
    (frequency_segment_summary["frequency_segment"] == "Low Frequency")
    & (frequency_segment_summary["sentiment_bucket"] == "Greed")
].iloc[0]
large_size_fear = size_segment_summary[
    (size_segment_summary["size_segment"] == "Large Size")
    & (size_segment_summary["sentiment_bucket"] == "Fear")
].iloc[0]
large_size_greed = size_segment_summary[
    (size_segment_summary["size_segment"] == "Large Size")
    & (size_segment_summary["sentiment_bucket"] == "Greed")
].iloc[0]
small_size_greed = size_segment_summary[
    (size_segment_summary["size_segment"] == "Small Size")
    & (size_segment_summary["sentiment_bucket"] == "Greed")
].iloc[0]
winner_greed = winner_segment_summary[
    (winner_segment_summary["winner_segment"] == "Winners")
    & (winner_segment_summary["sentiment_bucket"] == "Greed")
].iloc[0]
loser_fear = winner_segment_summary[
    (winner_segment_summary["winner_segment"] == "Losers")
    & (winner_segment_summary["sentiment_bucket"] == "Fear")
].iloc[0]

print("Insight 1")
print(
    f"High-frequency traders capture most of the Greed regime edge: median daily PnL rises from ${high_freq_fear['median_daily_pnl']:.2f} in Fear to ${high_freq_greed['median_daily_pnl']:.2f} in Greed, while low-frequency traders stay near flat in Greed at ${low_freq_greed['median_daily_pnl']:.2f}."
)
print()
print("Insight 2")
print(
    f"Large Size traders weaken when sentiment turns Greed: median daily PnL falls from ${large_size_fear['median_daily_pnl']:.2f} in Fear to ${large_size_greed['median_daily_pnl']:.2f} in Greed, while Small Size traders improve to ${small_size_greed['median_daily_pnl']:.2f}. This points to execution drag or weaker risk control when larger traders chase crowded moves."
)
print()
print("Insight 3")
print(
    f"Winner accounts behave like a different process, not just a better outcome: Winners post ${winner_greed['median_daily_pnl']:.2f} in Greed, while Losers are only at ${loser_fear['median_daily_pnl']:.2f} even in Fear. The edge appears to come from process discipline and regime fit rather than raw market direction alone."
)


## 9. Strategy Ideas

These recommendations are intentionally sharp and tied to the segment results.


In [ ]:
strategy_ideas = pd.DataFrame([
    {
        "market_regime": "Fear",
        "rule": "Reduce risk for low-frequency and weaker-PnL traders",
        "action": "Cap position size and require cleaner setups for Low Frequency or Loser-segment traders during Fear.",
        "why": "These groups do not earn enough median PnL in Fear to justify extra volatility exposure.",
    },
    {
        "market_regime": "Greed",
        "rule": "Keep fast trend-following active, but cut size for large notional traders",
        "action": "Allow High Frequency traders to stay active in Greed, but reduce notional size or tighten filters for Large Size accounts.",
        "why": "Greed benefits fast execution, but larger-size traders lose edge as crowding rises.",
    },
])

strategy_ideas


## 10. Export Tables and Summary


In [ ]:
quality_summary.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)
overall_sentiment_summary.to_csv(OUTPUT_DIR / "overall_sentiment_summary.csv", index=False)
fear_greed_comparison.to_csv(OUTPUT_DIR / "fear_greed_comparison.csv", index=False)
account_summary.to_csv(OUTPUT_DIR / "account_segment_summary.csv", index=False)
frequency_segment_summary.to_csv(OUTPUT_DIR / "frequency_segment_summary.csv", index=False)
size_segment_summary.to_csv(OUTPUT_DIR / "size_segment_summary.csv", index=False)
winner_segment_summary.to_csv(OUTPUT_DIR / "winner_segment_summary.csv", index=False)

report = f"""# Trader Performance vs Market Sentiment

## Methodology
- Loaded both raw datasets and checked shapes, duplicates, and missing values.
- Converted `Timestamp IST` to daily dates and aligned trader data with the Fear & Greed index.
- Built an all-traders trader-day panel across 32 accounts.
- Computed daily PnL, realized win rate, trades/day, position size, and long/short ratio.
- Used position size as a risk-taking proxy because a true leverage column is not available in the source file.
- Segmented traders into Frequent vs Infrequent, Large Size vs Small Size, and Winners vs Losers.

## Fear vs Greed Comparison
- Avg PnL: Fear ${fear_greed_comparison.loc[fear_greed_comparison['sentiment_bucket'] == 'Fear', 'avg_pnl'].iloc[0]:.2f} vs Greed ${fear_greed_comparison.loc[fear_greed_comparison['sentiment_bucket'] == 'Greed', 'avg_pnl'].iloc[0]:.2f}
- Avg realized win rate: Fear {fear_greed_comparison.loc[fear_greed_comparison['sentiment_bucket'] == 'Fear', 'avg_win_rate'].iloc[0]:.1%} vs Greed {fear_greed_comparison.loc[fear_greed_comparison['sentiment_bucket'] == 'Greed', 'avg_win_rate'].iloc[0]:.1%}
- Avg trades/day: Fear {fear_greed_comparison.loc[fear_greed_comparison['sentiment_bucket'] == 'Fear', 'avg_trades_per_day'].iloc[0]:.1f} vs Greed {fear_greed_comparison.loc[fear_greed_comparison['sentiment_bucket'] == 'Greed', 'avg_trades_per_day'].iloc[0]:.1f}

## Key Insights
1. High-frequency traders capture most of the Greed edge.
2. Large Size traders weaken when sentiment shifts to Greed, even though the overall regime is positive.
3. Winners and losers behave like different systems, suggesting process discipline matters more than raw market direction.

## Strategy Ideas
1. During Fear, reduce risk for low-frequency and weaker-PnL traders.
2. During Greed, keep fast traders active but cut size for large notional traders.
"""

(OUTPUT_DIR / "assignment_summary.md").write_text(report, encoding="utf-8")
print((OUTPUT_DIR / "assignment_summary.md").read_text(encoding="utf-8"))
